<a href="https://colab.research.google.com/github/SJH-official/Application_Developments/blob/main/%EB%8F%84%EC%84%9C%EA%B4%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# 1. 코랩 환경 필수 라이브러리 설치
!pip install -q gradio sqlalchemy

import gradio as gr
from datetime import datetime, timedelta
# 💡 아래 라인에 'ForeignKey'를 추가했습니다!
from sqlalchemy import create_engine, Column, Integer, String, Boolean, DateTime, Text, func, ForeignKey
from sqlalchemy.orm import sessionmaker, DeclarativeBase, relationship, joinedload

# ══════════════════════════════════════════
# [database.py 역할] 도서관 설계도 & 연결
# ══════════════════════════════════════════
DATABASE_URL = "sqlite:///./library.db"

engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)

class Base(DeclarativeBase):
    pass


# ══════════════════════════════════════════
# [models.py 역할] 책 분류 규칙 (테이블 정의)
# ══════════════════════════════════════════
class DBBook(Base):
    __tablename__ = "books"
    id           = Column(Integer, primary_key=True, index=True)
    title        = Column(String(200), nullable=False)
    author       = Column(String(100), nullable=False)
    isbn         = Column(String(20), unique=True, nullable=False, index=True)
    category     = Column(String(50), default="일반")
    total_copies = Column(Integer, default=1)
    available    = Column(Integer, default=1)
    description  = Column(Text, default="")
    created_at   = Column(DateTime, default=func.now())
    loans        = relationship("DBLoan", back_populates="book")

class DBMember(Base):
    __tablename__ = "members"
    id         = Column(Integer, primary_key=True, index=True)
    name       = Column(String(50), nullable=False)
    member_id  = Column(String(20), unique=True, nullable=False, index=True)
    phone      = Column(String(20), default="")
    email      = Column(String(100), default="")
    is_active  = Column(Boolean, default=True)
    created_at = Column(DateTime, default=func.now())
    loans      = relationship("DBLoan", back_populates="member")

class DBLoan(Base):
    __tablename__ = "loans"
    id          = Column(Integer, primary_key=True, index=True)
    book_id     = Column(Integer, ForeignKey("books.id", ondelete="CASCADE"), nullable=False)
    member_id   = Column(Integer, ForeignKey("members.id", ondelete="CASCADE"), nullable=False)
    loan_date   = Column(DateTime, default=func.now())
    due_date    = Column(DateTime, nullable=False)
    return_date = Column(DateTime, nullable=True)
    is_returned = Column(Boolean, default=False)
    book        = relationship("DBBook", back_populates="loans")
    member      = relationship("DBMember", back_populates="loans")


# ══════════════════════════════════════════
# [main.py 비즈니스 로직] 도서관 서비스 기능
# ══════════════════════════════════════════
def startup():
    """도서관 개관: 테이블 생성 및 샘플 데이터 삽입"""
    Base.metadata.create_all(bind=engine)
    db = SessionLocal()
    try:
        if db.query(DBBook).count() > 0:
            return
        sample_books = [
            DBBook(title="파이썬 완전정복", author="홍길동", isbn="9788901000001", category="IT", total_copies=3, available=3),
            DBBook(title="FastAPI 마스터", author="김철수", isbn="9788901000002", category="IT", total_copies=2, available=2),
            DBBook(title="채식주의자", author="한강", isbn="9788901000004", category="문학", total_copies=4, available=4),
            DBBook(title="코스모스", author="칼 세이건", isbn="9788901000006", category="과학", total_copies=2, available=2),
        ]
        db.add_all(sample_books)
        sample_members = [
            DBMember(name="박민준", member_id="M001", phone="010-1234-5678", email="minjun@email.com"),
            DBMember(name="이서연", member_id="M002", phone="010-2345-6789", email="seoyeon@email.com"),
        ]
        db.add_all(sample_members)
        db.commit()
    finally:
        db.close()

def add_book(title, author, isbn, category, copies, description):
    if not all([title.strip(), author.strip(), isbn.strip()]):
        return "❌ 제목, 저자, ISBN은 필수 입력 항목입니다."
    db = SessionLocal()
    try:
        if db.query(DBBook).filter(DBBook.isbn == isbn.strip()).first():
            return f"❌ ISBN '{isbn}'이 이미 등록되어 있습니다."
        book = DBBook(title=title.strip(), author=author.strip(), isbn=isbn.strip(), category=category, total_copies=int(copies), available=int(copies), description=description.strip())
        db.add(book)
        db.commit()
        return f"✅ 도서 등록 완료!\n📚 [{category}] {title} (ISBN: {isbn})"
    finally:
        db.close()

def get_books(search="", category_filter="전체"):
    db = SessionLocal()
    try:
        q = db.query(DBBook)
        if search.strip():
            kw = f"%{search.strip()}%"
            q = q.filter(DBBook.title.like(kw) | DBBook.author.like(kw) | DBBook.isbn.like(kw))
        if category_filter != "전체":
            q = q.filter(DBBook.category == category_filter)
        books = q.order_by(DBBook.id.desc()).all()
        if not books: return [["검색 결과 없음", "", "", "", "", ""]]
        return [[b.id, b.title, b.author, b.category, f"{b.available}/{b.total_copies}", "대출가능" if b.available > 0 else "대출불가"] for b in books]
    finally:
        db.close()

def add_member(name, member_id, phone, email):
    if not all([name.strip(), member_id.strip()]):
        return "❌ 이름과 회원번호는 필수 입력 항목입니다."
    db = SessionLocal()
    try:
        if db.query(DBMember).filter(DBMember.member_id == member_id.strip()).first():
            return f"❌ 회원번호 '{member_id}'이 이미 존재합니다."
        member = DBMember(name=name.strip(), member_id=member_id.strip(), phone=phone.strip(), email=email.strip())
        db.add(member)
        db.commit()
        return f"✅ 회원 등록 완료!\n👤 {name} (회원번호: {member_id})"
    finally:
        db.close()

def get_members(search=""):
    db = SessionLocal()
    try:
        q = db.query(DBMember).options(joinedload(DBMember.loans))
        if search.strip():
            kw = f"%{search.strip()}%"
            q = q.filter(DBMember.name.like(kw) | DBMember.member_id.like(kw))
        members = q.order_by(DBMember.id.desc()).all()
        if not members: return [["검색 결과 없음", "", "", "", "", ""]]
        rows = []
        for m in members:
            active_loans = sum(1 for l in m.loans if not l.is_returned)
            rows.append([m.id, m.member_id, m.name, m.phone, "활성" if m.is_active else "비활성", f"대출중 {active_loans}권"])
        return rows
    finally:
        db.close()

def loan_book(book_id_str, member_id_str, due_days):
    if not book_id_str.strip() or not member_id_str.strip():
        return "❌ 도서 ID와 회원번호를 입력해주세요."
    db = SessionLocal()
    try:
        book = db.query(DBBook).filter(DBBook.id == int(book_id_str)).first()
        member = db.query(DBMember).filter(DBMember.member_id == member_id_str.strip()).first()
        if not book: return "❌ 해당 도서를 찾을 수 없습니다."
        if book.available <= 0: return f"❌ 재고가 없습니다."
        if not member: return "❌ 해당 회원을 찾을 수 없습니다."

        existing = db.query(DBLoan).filter(DBLoan.book_id == book.id, DBLoan.member_id == member.id, DBLoan.is_returned == False).first()
        if existing: return f"❌ 이미 이 책을 대출 중입니다."

        due = datetime.now() + timedelta(days=int(due_days))
        loan = DBLoan(book_id=book.id, member_id=member.id, due_date=due)
        book.available -= 1
        db.add(loan)
        db.commit()
        return f"✅ 대출 완료!\n📖 {book.title} → 👤 {member.name}\n📅 반납 예정일: {due.strftime('%Y-%m-%d')}"
    except ValueError: return "❌ 숫자로 입력해주세요."
    finally: db.close()

def return_book(loan_id_str):
    if not loan_id_str.strip():
        return "❌ 대출 ID를 입력해주세요."
    db = SessionLocal()
    try:
        loan = db.query(DBLoan).options(joinedload(DBLoan.book), joinedload(DBLoan.member)).filter(DBLoan.id == int(loan_id_str)).first()
        if not loan: return "❌ 대출 기록이 없습니다."
        if loan.is_returned: return "❌ 이미 반납된 도서입니다."

        loan.is_returned = True
        loan.return_date = datetime.now()
        loan.book.available += 1
        overdue = datetime.now() > loan.due_date
        msg = "⚠️ 연체 반납" if overdue else "✅ 반납 완료"
        res_msg = f"{msg}!\n📖 {loan.book.title} ← 👤 {loan.member.name}"
        if overdue:
            res_msg += f"\n🔴 연체일수: {(datetime.now() - loan.due_date).days}일"
        db.commit()
        return res_msg
    except ValueError: return "❌ 올바른 ID를 입력하세요."
    finally: db.close()

def get_loans(show_all=False):
    db = SessionLocal()
    try:
        q = db.query(DBLoan).options(joinedload(DBLoan.book), joinedload(DBLoan.member))
        if not show_all: q = q.filter(DBLoan.is_returned == False)
        loans = q.order_by(DBLoan.loan_date.desc()).all()
        if not loans: return [["대출 기록 없음", "", "", "", "", ""]]
        rows = []
        for l in loans:
            overdue = not l.is_returned and datetime.now() > l.due_date
            status = "반납완료" if l.is_returned else ("🔴연체" if overdue else "대출중")
            rows.append([l.id, l.book.title, f"{l.member.name} ({l.member.member_id})", l.loan_date.strftime("%Y-%m-%d"), l.due_date.strftime("%Y-%m-%d"), status])
        return rows
    finally:
        db.close()

def get_dashboard():
    db = SessionLocal()
    try:
        return f"📚 총 도서: {db.query(DBBook).count()}종  |  👥 총 회원: {db.query(DBMember).count()}명  |  📋 대출 중: {db.query(DBLoan).filter(DBLoan.is_returned == False).count()}건"
    finally:
        db.close()

# ══════════════════════════════════════════
# [Gradio UI] 화면 구성 및 레이아웃
# ══════════════════════════════════════════
CATEGORIES = ["전체", "IT", "문학", "과학", "역사", "예술", "기타"]
BOOK_CATEGORIES = ["IT", "문학", "과학", "역사", "예술", "기타"]
BOOK_HEADERS  = ["ID", "제목", "저자", "카테고리", "재고(가능/전체)", "상태"]
MEMBER_HEADERS = ["ID", "회원번호", "이름", "전화번호", "상태", "대출현황"]
LOAN_HEADERS  = ["대출ID", "도서명", "회원", "대출일", "반납예정일", "상태"]

with gr.Blocks(title="📚 도서관 대여 관리 시스템", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📚 도서관 대여 관리 시스템 ")
    dash_text = gr.Textbox(label="도서관 현황 대시보드", interactive=False)

    with gr.Tabs():
        with gr.Tab("📖 도서 관리"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### ➕ 신규 도서 등록")
                    b_title = gr.Textbox(label="제목 *")
                    b_author = gr.Textbox(label="저자 *")
                    b_isbn = gr.Textbox(label="ISBN *")
                    b_cat = gr.Dropdown(BOOK_CATEGORIES, label="카테고리", value="IT")
                    b_copies = gr.Number(label="수량", value=1, precision=0)
                    b_desc = gr.Textbox(label="설명")
                    b_add_btn = gr.Button("등록", variant="primary")
                    b_result = gr.Textbox(label="결과", interactive=False)
                with gr.Column(scale=2):
                    gr.Markdown("### 🔍 도서 목록 및 검색")
                    with gr.Row():
                        b_search = gr.Textbox(label="검색어", scale=2)
                        b_cat_flt = gr.Dropdown(CATEGORIES, label="필터", value="전체", scale=1)
                        b_srch_btn = gr.Button("검색", scale=1)
                    b_table = gr.Dataframe(headers=BOOK_HEADERS, interactive=False)

            b_add_btn.click(add_book, [b_title, b_author, b_isbn, b_cat, b_copies, b_desc], b_result).then(get_books, [], b_table).then(get_dashboard, [], dash_text)
            b_srch_btn.click(get_books, [b_search, b_cat_flt], b_table)

        with gr.Tab("👥 회원 관리"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### ➕ 신규 회원 등록")
                    m_name = gr.Textbox(label="이름 *")
                    m_id = gr.Textbox(label="회원번호 * (예: M003)")
                    m_phone = gr.Textbox(label="전화번호")
                    m_email = gr.Textbox(label="이메일")
                    m_add_btn = gr.Button("등록", variant="primary")
                    m_result = gr.Textbox(label="결과", interactive=False)
                with gr.Column(scale=2):
                    gr.Markdown("### 🔍 회원 목록")
                    with gr.Row():
                        m_search = gr.Textbox(label="이름/회원번호 검색", scale=3)
                        m_srch_btn = gr.Button("검색", scale=1)
                    m_table = gr.Dataframe(headers=MEMBER_HEADERS, interactive=False)

            m_add_btn.click(add_member, [m_name, m_id, m_phone, m_email], m_result).then(get_members, [], m_table).then(get_dashboard, [], dash_text)
            m_srch_btn.click(get_members, [m_search], m_table)

        with gr.Tab("📋 대출 / 반납"):
            with gr.Row():
                with gr.Column():
                    gr.Markdown("### 📤 도서 대출")
                    l_b_id = gr.Textbox(label="도서 ID")
                    l_m_id = gr.Textbox(label="회원번호")
                    l_days = gr.Slider(7, 30, value=14, step=1, label="대출 기간(일)")
                    l_btn = gr.Button("대출 실행", variant="primary")
                    l_result = gr.Textbox(label="결과", interactive=False)
                with gr.Column():
                    gr.Markdown("### 📥 도서 반납")
                    r_l_id = gr.Textbox(label="대출 기록 ID")
                    r_btn = gr.Button("반납 실행", variant="secondary")
                    r_result = gr.Textbox(label="결과", interactive=False)

            gr.Markdown("### 📊 실시간 대출 현황")
            show_all_chk = gr.Checkbox(label="반납 완료 내역 포함", value=False)
            loan_table = gr.Dataframe(headers=LOAN_HEADERS, interactive=False)

            l_btn.click(loan_book, [l_b_id, l_m_id, l_days], l_result).then(get_loans, [show_all_chk], loan_table).then(get_dashboard, [], dash_text)
            r_btn.click(return_book, [r_l_id], r_result).then(get_loans, [show_all_chk], loan_table).then(get_dashboard, [], dash_text)
            show_all_chk.change(get_loans, [show_all_chk], loan_table)

    # 초기 데이터 로드 체인
    demo.load(get_dashboard, outputs=dash_text)
    demo.load(get_books, inputs=[], outputs=b_table)
    demo.load(get_members, inputs=[], outputs=m_table)
    demo.load(get_loans, inputs=[show_all_chk], outputs=loan_table)

# ══════════════════════════════════════════
# 프로그램 구동 (Colab 최적화 설정)
# ══════════════════════════════════════════
if __name__ == "__main__":
    startup()
    demo.launch(inline=True, share=True)

/tmp/ipykernel_7975/1532931702.py:225: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="📚 도서관 대여 관리 시스템", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://47dd8fb1cd28323723.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
